# SpeechLens — T4 validation run

Authoritative numbers for this project come from a **free cloud T4**
(Colab / Kaggle) running `large-v3` at `compute_type=float16`. There is no
local NVIDIA GPU on any dev machine, so this notebook *is* the measurement
rig: it clones the repo, installs the stack, re-checks the offline test
invariant, then runs all three validation studies and prints a block you
paste straight into `docs/VALIDATION.md`.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`, then
`Runtime -> Run all`. Expect ~15-25 min end to end; most of it is the
one-time `large-v3` weight download (~3 GB) and the bench sweep.

What each study is for:

| Study | Question it answers |
|---|---|
| `smoke` | Does LID + the decode path work across 5 languages at all? Plumbing, not accuracy. |
| `snr` | How gracefully do WER and LID degrade as noise rises — and do confidence flags climb *before* the transcript goes bad? |
| `bench` | Real-time factor per model size, for the portfolio RTF chip. |

The SNR study is run twice, with and without `--denoise`, to A/B the
spectral-gating stage (prior: denoise loses above ~5 dB SNR).

In [ ]:
#@title Step 1 — confirm we actually have a GPU (and which one)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import subprocess, sys
try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True, text=True, check=True).stdout.strip()
except Exception:
    gpu = ""

if not gpu:
    sys.exit("No GPU visible. Runtime -> Change runtime type -> T4 GPU, "
             "then re-run. Numbers from a CPU runtime are NOT the "
             "authoritative T4 numbers and must not be recorded as such.")

print(f"\nGPU: {gpu}")
if "T4" not in gpu:
    print("WARNING: this is not a T4. The run will still work, but record "
          "the real device name in docs/VALIDATION.md — never label these "
          "numbers 'T4'.")

In [ ]:
#@title Step 2 — clone the repo
import os

REPO = "https://github.com/shaostassen/voice_detection_using_machine_learning.git"
WORKDIR = "/content/voice_detection_using_machine_learning"

if not os.path.isdir(WORKDIR):
    !git clone -q $REPO $WORKDIR
else:
    !git -C $WORKDIR pull -q --ff-only

%cd $WORKDIR
!git log --oneline -1

In [ ]:
#@title Step 3 — install the stack
# espeak-ng: required by the smoke study and by tests/test_silero.py.
!apt-get -qq update && apt-get -qq install -y espeak-ng > /dev/null

# denoise extra is needed for the --denoise arm of the SNR A/B.
%pip install -q -e ".[dev,denoise]"

# CTranslate2's CUDA path wants cuBLAS + cuDNN 9 for CUDA 12. Colab images
# do not always ship cuDNN 9, and the failure mode is an opaque libcudnn
# load error at first GPU inference, so install and point the loader at them.
%pip install -q nvidia-cublas-cu12 "nvidia-cudnn-cu12>=9"

import os, glob, nvidia

# The nvidia-* wheels ship *implicit namespace packages*: nvidia/cudnn/lib
# has no __init__.py, so nvidia.cudnn.lib.__file__ is None and dirname()
# raises. Walk __path__ instead and keep every nvidia/*/lib that actually
# contains shared objects.
lib_dirs = []
for root in list(nvidia.__path__):
    for d in sorted(glob.glob(os.path.join(root, "*", "lib"))):
        if glob.glob(os.path.join(d, "*.so*")):
            lib_dirs.append(d)

# Set it in os.environ (not just the shell) so every subprocess below
# inherits it — the dynamic linker reads LD_LIBRARY_PATH at process start,
# which is exactly when each validate.py run begins.
prev = os.environ.get("LD_LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = ":".join(lib_dirs + ([prev] if prev else []))
print("\n".join(lib_dirs) or "no nvidia lib dirs found")

assert any("cudnn" in d for d in lib_dirs), \
    "cuDNN libs not found — CTranslate2's CUDA path will fail to load"

# Datasets is only used to fetch the labeled validation clip in step 5.
%pip install -q datasets

In [ ]:
#@title Step 4 — re-check the offline test invariant on this box
# pytest must pass with no network and no model weights. If this fails,
# stop: the studies below are not measuring what you think they are.
!HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 python -m pytest -q

## Step 5 — the validation clip

The SNR sweep needs **real human speech with a ground-truth transcript**,
otherwise its WER numbers are not publishable. Default source is a few
LibriSpeech utterances (exact reference text, ~16 kHz read English).

`normalize_text` in `speechlens/metrics.py` lowercases and strips
punctuation, so LibriSpeech's uppercase unpunctuated references compare
fairly against Whisper's cased, punctuated output. The one known residual
is spelled-out numbers vs. digits — a handful of tokens, not corrected here.

Set `SOURCE` below:

- `"librispeech"` — labeled real speech. **Use this for any number you publish.**
- `"upload"` — your own clip; you must supply `REFERENCE` yourself.
- `"espeak"` — synthetic TTS fallback. Plumbing check only; its WER is
  *not* a publishable accuracy figure.

In [ ]:
#@title Step 5 — build the clip
SOURCE = "librispeech"  #@param ["librispeech", "upload", "espeak"]
N_UTTERANCES = 3        #@param {type:"integer"}

import numpy as np, soundfile as sf
from pathlib import Path

CLIP = Path("/content/validation_clip.wav")


def from_librispeech():
    from datasets import load_dataset
    ds = load_dataset("hf-internal-testing/librispeech_asr_dummy",
                      "clean", split="validation")
    rows = [ds[i] for i in range(min(N_UTTERANCES, len(ds)))]
    y = np.concatenate([np.asarray(r["audio"]["array"], dtype=np.float32)
                        for r in rows])
    sr = int(rows[0]["audio"]["sampling_rate"])
    ref = " ".join(r["text"].strip() for r in rows)
    return y, sr, ref, True


def from_upload():
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    import subprocess
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", name,
                    "-ar", "16000", "-ac", "1", str(CLIP)], check=True)
    y, sr = sf.read(str(CLIP), dtype="float32")
    return y, sr, REFERENCE_OVERRIDE, bool(REFERENCE_OVERRIDE)


def from_espeak():
    import subprocess
    text = ("The quick brown fox jumps over the lazy dog, while the "
            "calibration microphone records everything in the room.")
    subprocess.run(["espeak-ng", "-s", "150", "-w", str(CLIP), text],
                   check=True)
    y, sr = sf.read(str(CLIP), dtype="float32")
    return y, sr, text, False


REFERENCE_OVERRIDE = ""  # fill in if SOURCE == "upload"

builders = {"librispeech": from_librispeech, "upload": from_upload,
            "espeak": from_espeak}
try:
    audio, SR, REFERENCE, PUBLISHABLE = builders[SOURCE]()
except Exception as e:
    print(f"{SOURCE} source failed ({e}); falling back to espeak TTS.")
    audio, SR, REFERENCE, PUBLISHABLE = from_espeak()
    SOURCE = "espeak"

sf.write(str(CLIP), audio, SR)
DURATION = len(audio) / SR
print(f"source={SOURCE}  duration={DURATION:.1f}s  sr={SR}")
print(f"publishable WER: {PUBLISHABLE}")
print(f"reference: {REFERENCE[:300]}{'...' if len(REFERENCE) > 300 else ''}")

if not PUBLISHABLE:
    print("\nNOTE: synthetic or unlabeled audio. Error rates below validate "
          "plumbing only — do NOT put them in the README or portfolio.")

from IPython.display import Audio, display
display(Audio(str(CLIP)))

## Step 6 — run the studies

Each run is captured verbatim. `validate.py` prints a provenance banner
(hardware / model / device / compute_type / date / raw command) as its first
two lines, so every table below carries the run that produced it.

In [ ]:
#@title Step 6 — study runner
import subprocess, sys, time
from pathlib import Path

MODEL = "large-v3"        #@param {type:"string"}
COMPUTE_TYPE = "float16"  #@param ["float16", "int8_float16", "int8", "float32"]

RUNS = {}
LOGDIR = Path("/content/validation_runs"); LOGDIR.mkdir(exist_ok=True)


def run_study(name, args):
    """Run validate.py, stream output, keep it verbatim for the writeup."""
    cmd = [sys.executable, "scripts/validate.py", *args,
           "--device", "cuda", "--compute-type", COMPUTE_TYPE]
    print(f"$ {' '.join(cmd)}\n")
    t0 = time.perf_counter()
    p = subprocess.run(cmd, capture_output=True, text=True)
    out = p.stdout + (("\n[stderr]\n" + p.stderr) if p.returncode else "")
    print(out)
    print(f"[{name}: exit={p.returncode} wall={time.perf_counter()-t0:.0f}s]")
    RUNS[name] = out
    (LOGDIR / f"{name}.txt").write_text(out)
    return p.returncode

### Study 1 — multilingual smoke (LID + plumbing)

espeak synthesizes a known sentence in 5 languages; we check the detected
language code and report error rates. Nonzero error is expected — espeak
audio is robotic. The number that matters here is **LID hits: n/5**.

In [ ]:
run_study("smoke", ["smoke", "--model", MODEL])

### Study 2 — SNR sweep (the core robustness result)

White noise mixed at falling SNR: clean, 20, 10, 5, 0, -5 dB, seeded so the
ladder is reproducible. Three things to watch, per the runbook:

1. **LID stability** as noise rises — chunk voting should hold the correct
   language further down the ladder than a single window would.
2. **Graceful WER degradation** rather than a cliff into hallucinated text.
3. **Flagged-segment counts climbing _before_ the transcript goes bad** —
   flags leading errors is the whole point of the confidence gate.

In [ ]:
ref_args = ["--ref", REFERENCE] if REFERENCE else []
run_study("snr", ["snr", str(CLIP), "--model", MODEL, *ref_args])

In [ ]:
#@title Same sweep with the spectral-gating denoise stage ON
run_study("snr_denoise",
          ["snr", str(CLIP), "--model", MODEL, *ref_args, "--denoise"])

### Study 3 — RTF bench across model sizes

Real-time factor = processing time / audio duration; lower is better,
`1/rtf` is the "x faster than realtime" figure. Model load happens in the
constructor and is excluded from the timing, so this measures decode
throughput. This produces the portfolio RTF chip.

In [ ]:
run_study("bench", ["bench", str(CLIP), "--models",
                    "base,small,distil-large-v3,large-v3"])

## Step 7 — assemble the `docs/VALIDATION.md` entry

Copy the block below into `docs/VALIDATION.md`. It carries hardware, model,
compute_type, date and the raw command for every study, which is the
condition for a number being allowed to exist in this project.

In [ ]:
#@title Step 7 — emit the writeup block
import datetime, textwrap

gpu_name = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True).stdout.strip().splitlines()[0]

header = textwrap.dedent(f"""\
    ## Run — {datetime.date.today().isoformat()}

    | field | value |
    |---|---|
    | hardware | {gpu_name} (Colab) |
    | model | {MODEL} |
    | compute_type | {COMPUTE_TYPE} |
    | audio source | {SOURCE} ({DURATION:.1f}s) |
    | publishable WER | {PUBLISHABLE} |
    | notebook | `notebooks/validate_t4.ipynb` |
    """)

blocks = [header]
for name in ("smoke", "snr", "snr_denoise", "bench"):
    if name in RUNS:
        blocks.append(f"### {name}\n\n```\n{RUNS[name].strip()}\n```\n")

writeup = "\n".join(blocks)
Path("/content/VALIDATION_entry.md").write_text(writeup)
print(writeup)

In [ ]:
#@title Download the raw logs + writeup
!cd /content && zip -qr validation_t4_logs.zip validation_runs VALIDATION_entry.md
from google.colab import files
files.download("/content/validation_t4_logs.zip")

## After this notebook

1. Paste the block above into `docs/VALIDATION.md`.
2. Fill the two pending portfolio chips in `docs/portfolio-section.html`
   (RTF from the bench table, WER at -5 dB from the SNR table) — **only if
   `publishable WER` was True**.
3. Add a Results section to the README from these numbers plus the 9950X
   CPU baseline (task 4 in the task queue).

Keep the pending chips pending until real numbers exist. No fabricated
metrics, anywhere.